# Module 06 — GRPO (Group Relative Policy Optimisation)

**Goal**: Train a model to solve math problems using *verifiable rewards* — no reward model needed.

GRPO is the algorithm behind DeepSeek-R1 and is directly applicable to the Kaggle competition (test pass/fail = binary verifiable reward).

**What you will build**:
1. A math dataset with ground-truth answers
2. A verifiable reward function (answer correctness)
3. Run `GRPOTrainer` for 80 steps
4. Watch accuracy improve

In [ ]:
# Cell 1 — Install
!pip install -q transformers datasets trl peft accelerate torch

## 1. Why GRPO?

Both PPO and DPO require a *learned* reward model trained on human preferences.
This introduces noise: the reward model may be wrong, and the policy can exploit its weaknesses.

**Verifiable rewards** sidestep this entirely. For tasks where correctness is objectively checkable:

```
Math:      Does the final answer match the ground truth?          → 1.0 / 0.0
Code:      Do the unit tests pass?                                → 1.0 / 0.0
Format:    Is the output valid JSON?                              → 1.0 / 0.0
```

**GRPO** (Shao et al., 2024) extends PPO with a key change:

Instead of a learned value function (critic), it generates **G responses per prompt**
and uses the *group average* reward as the baseline:

```
Advantage_i = (r_i - mean(r_1..r_G)) / std(r_1..r_G)
```

This eliminates the value head entirely — simpler, cheaper, and the baseline is exact
rather than estimated.

**Kaggle connection**: In the Gemma 4 Agent competition, reward = `test_pass / total_tests`
(binary per task). That is a verifiable reward — GRPO applies directly.

In [ ]:
# Cell 2 — Imports
import torch, re
from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import Dataset
from trl import GRPOTrainer, GRPOConfig
from peft import LoraConfig, TaskType, get_peft_model

MODEL_ID  = "HuggingFaceTB/SmolLM2-135M-Instruct"
DEVICE    = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE     = torch.bfloat16
MAX_STEPS = 80
G         = 4   # number of responses sampled per prompt
print(f"Device: {DEVICE}  |  G={G} responses per prompt")

## 2. Math dataset with verifiable answers

In [ ]:
# Cell 3 — Inline math problems
# Format: prompt + ground-truth answer the model should output as \boxed{answer}
MATH_DATA = [
    {"prompt": "What is 15 + 27?", "answer": "42"},
    {"prompt": "What is 8 × 7?", "answer": "56"},
    {"prompt": "What is 144 ÷ 12?", "answer": "12"},
    {"prompt": "What is 2^10?", "answer": "1024"},
    {"prompt": "What is the square root of 169?", "answer": "13"},
    {"prompt": "What is 17 × 13?", "answer": "221"},
    {"prompt": "What is 1000 - 387?", "answer": "613"},
    {"prompt": "A rectangle has width 8 and height 6. What is its area?", "answer": "48"},
    {"prompt": "What is 25% of 200?", "answer": "50"},
    {"prompt": "What is 3^5?", "answer": "243"},
    {"prompt": "What is 7! (7 factorial)?", "answer": "5040"},
    {"prompt": "A train travels at 60 mph for 2.5 hours. How many miles does it cover?", "answer": "150"},
    {"prompt": "What is the sum of all integers from 1 to 10?", "answer": "55"},
    {"prompt": "What is 999 × 9?", "answer": "8991"},
    {"prompt": "What is 5^3 + 3^5?", "answer": "368"},
    {"prompt": "A circle has radius 5. What is its area? (Use π ≈ 3.14, round to 2 decimal places)", "answer": "78.5"},
    {"prompt": "What is GCD(48, 18)?", "answer": "6"},
    {"prompt": "What is LCM(4, 6)?", "answer": "12"},
    {"prompt": "If x + 5 = 13, what is x?", "answer": "8"},
    {"prompt": "What is 2^8?", "answer": "256"},
] * 3   # 60 examples

# Format with a system prompt that teaches the model to box its answer
SYSTEM = (
    "You are a math assistant. Solve the problem step by step. "
    "Put your final numeric answer inside \\boxed{} like this: \\boxed{42}"
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token

def format_prompt(row):
    msgs = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": row["prompt"]},
    ]
    return {
        "prompt": tokenizer.apply_chat_template(
            msgs, tokenize=False, add_generation_prompt=True
        ),
        "answer": row["answer"],
    }

math_ds = Dataset.from_list(MATH_DATA).map(format_prompt)
print(f"Dataset: {len(math_ds)} math problems")
print("\nExample prompt (last 200 chars):")
print("..." + math_ds[0]["prompt"][-200:])

## 3. Verifiable reward function

This is the core of GRPO: a function that takes the model's response and returns a scalar reward with no learned model involved.

In [ ]:
# Cell 4 — Reward function
def extract_answer(text):
    """Extract the number from \\boxed{...} if present, else try the last number."""
    # Try \boxed{...}
    boxed = re.findall(r'\\boxed\{([^}]+)\}', text)
    if boxed:
        return boxed[-1].strip()
    # Fallback: last number in the response
    nums = re.findall(r'-?\d+\.?\d*', text)
    return nums[-1] if nums else None

def reward_fn(completions, answer, **kwargs):
    """
    Returns a list of rewards (one per completion).
    1.0  = exact match
    0.0  = wrong or no answer found
    """
    rewards = []
    for completion, gt in zip(completions, answer):
        pred = extract_answer(completion)
        correct = (pred is not None and pred.strip() == str(gt).strip())
        rewards.append(1.0 if correct else 0.0)
    return rewards

# Quick test of the reward function
test_completions = [
    "Let me think... 15 + 27 = 42. So \\boxed{42}",
    "The answer is 42",
    "I think it's \\boxed{43}",
    "42",
]
test_answers = ["42"] * 4
print("Reward function test:")
for comp, r in zip(test_completions, reward_fn(test_completions, test_answers)):
    print(f"  '{comp[:50]}'  →  reward={r}")

## 4. Measure baseline accuracy (before GRPO)

In [ ]:
# Cell 5 — Baseline accuracy
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=DTYPE, device_map=DEVICE
)
model.eval()

def eval_accuracy(model, tok, dataset, n=20):
    correct = 0
    for row in dataset.select(range(n)):
        inputs = tok(row["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            ids = model.generate(**inputs, max_new_tokens=80,
                                 temperature=0.0, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        response = tok.decode(ids[0][inputs["input_ids"].shape[-1]:],
                              skip_special_tokens=True)
        pred = extract_answer(response)
        if pred and pred.strip() == str(row["answer"]).strip():
            correct += 1
    return correct / n

baseline_acc = eval_accuracy(model, tokenizer, math_ds)
print(f"Baseline accuracy (greedy, n=20): {baseline_acc:.1%}")

## 5. Train with GRPOTrainer

In [ ]:
# Cell 6 — LoRA
lora_cfg = LoraConfig(
    r=8, lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05, bias="none",
    task_type=TaskType.CAUSAL_LM,
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

In [ ]:
# Cell 7 — GRPOConfig
grpo_config = GRPOConfig(
    output_dir="/tmp/grpo_math",
    max_steps=MAX_STEPS,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=5e-5,
    lr_scheduler_type="cosine",
    warmup_steps=10,
    num_generations=G,           # G responses per prompt for group baseline
    max_new_tokens=80,
    temperature=0.9,
    logging_steps=10,
    bf16=True,
    report_to="none",
)

In [ ]:
# Cell 8 — Train
grpo_trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    reward_funcs=reward_fn,      # our verifiable reward
    train_dataset=math_ds,
    processing_class=tokenizer,
)

print(f"Starting GRPO training  (G={G}, {MAX_STEPS} steps)...")
grpo_trainer.train()
print("Done!")

In [ ]:
# Cell 9 — Post-GRPO accuracy
merged = model.merge_and_unload()
merged.eval()

post_acc = eval_accuracy(merged, tokenizer, math_ds)
print(f"Baseline accuracy : {baseline_acc:.1%}")
print(f"Post-GRPO accuracy: {post_acc:.1%}")
print(f"Improvement       : {(post_acc - baseline_acc):+.1%}")

In [ ]:
# Cell 10 — Loss & reward curves
import matplotlib.pyplot as plt

log = grpo_trainer.state.log_history
steps   = [x["step"] for x in log if "loss" in x]
losses  = [x["loss"] for x in log if "loss" in x]
rewards = [x.get("reward", x.get("rewards/mean", 0)) for x in log if "loss" in x]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(steps, losses, color="#2a9d8f", linewidth=2)
ax1.set_title("GRPO Loss"); ax1.set_xlabel("Step"); ax1.grid(alpha=0.3)

ax2.plot(steps, rewards, color="#e76f51", linewidth=2)
ax2.set_title("Mean Reward (answer accuracy)"); ax2.set_xlabel("Step")
ax2.set_ylim(0, 1.05); ax2.axhline(y=baseline_acc, color="#264653",
    linestyle="--", label=f"Baseline {baseline_acc:.1%}")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("/tmp/grpo_curves.png", dpi=120)
plt.show()

## 6. Connection to the Kaggle competition

The Gemma 4 Developer Agent competition uses exactly this paradigm:

```python
# Competition reward function (conceptual)
def kaggle_reward(completion, test_suite, **kwargs):
    patch = extract_patch(completion)
    apply_patch(patch)
    results = run_tests(test_suite)
    return results["passed"] / results["total"]   # 0.0 – 1.0
```

Your curriculum path for the competition:

| Module | What it builds |
|--------|---------------|
| 02 SFT | Train on (issue → patch) demonstrations from git history |
| 05 DPO | Rank patches by test pass rate: preferred > rejected |
| 06 GRPO | Use test pass rate as verifiable reward — **this module** |

## ✅ Summary

| Concept | Detail |
|---------|--------|
| **Verifiable reward** | Correctness check — no learned reward model needed |
| **Group sampling (G)** | G responses per prompt; group mean = advantage baseline |
| **No value head** | Simpler than PPO — baseline from rewards, not a critic network |
| **Applications** | Math reasoning, code generation, structured output, agentic tasks |

**Congratulations — you've completed the core curriculum!**

The full post-training pipeline you now understand:

```
Pre-training → SFT (M02) → Reward Model (M03) → PPO (M04)
                                              → DPO (M05)
                                              → GRPO (M06)  ← cutting edge
```

> 💡 For the Kaggle competition: start with GRPO + test-pass reward on the four target repos.
> The binary reward signal is clean and strong — exactly the setting where GRPO excels.